In [7]:
import numpy as np
from scipy.stats import wasserstein_distance
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import os
import yaml
import pandas as pd
from pathlib import Path

In [ ]:
import os
from pathlib import Path

import pandas as pd
import yaml
from scipy.stats import wasserstein_distance


config_path = Path.cwd()
infer_path = os.path.join(os.path.dirname(os.path.dirname(config_path)), "inference.yaml")
with open(infer_path, "r") as file:
    config = yaml.safe_load(file)

features = config["data_drift"]["drift_features"]
min_ref_samples_for_iqr = config["data_drift"]["min_ref_samples_for_iqr"]
min_ref_samples_for_drift = config["data_drift"]["min_ref_samples_for_drift"]

drift_scores = {}
FEATURE_COL = "FeatureType"
SEASON_COL = "SeasonBin"
HOUR_COL = "HourBin"
DAYTYPE_COL = "DayType"
HOLIDAY_COL = "IsHoliday"
VALUE_COL = "Value"


def group_features(reference_data, production_data):
    temp_df_production = production_data[
        production_data[FEATURE_COL].str.contains("temperature", case=False, na=False)
    ]
    lag_df_production = production_data[
        production_data[FEATURE_COL].str.contains("lag", case=False, na=False)
    ]
    temp_df_reference = reference_data[
        reference_data[FEATURE_COL].str.contains("temperature", case=False, na=False)
    ]
    lag_df_reference = reference_data[
        reference_data[FEATURE_COL].str.contains("lag", case=False, na=False)
    ]
    return temp_df_reference, lag_df_reference, temp_df_production, lag_df_production


def group_lag_features(reference_data, production_data, lag_val):
    lag_df_reference = reference_data[reference_data[FEATURE_COL] == lag_val]
    lag_df_production = production_data[production_data[FEATURE_COL] == lag_val]
    return lag_df_reference, lag_df_production


def static_reference_drift(static_data, production_data):
    required_columns = {
        FEATURE_COL,
        SEASON_COL,
        HOUR_COL,
        DAYTYPE_COL,
        HOLIDAY_COL,
        VALUE_COL,
    }
    missing_static = required_columns - set(static_data.columns)
    missing_production = required_columns - set(production_data.columns)
    if missing_static:
        raise ValueError(f"Missing columns in static data: {missing_static}")
    if missing_production:
        raise ValueError(f"Missing columns in production data: {missing_production}")

    unknown = {
        feature
        for feature in production_data[FEATURE_COL].dropna().astype(str).unique()
        if feature.lower() not in set(features)
    }
    if unknown:
        raise ValueError(f"Unknown features in production data: {unknown}")

    (
        temp_df_reference,
        lag_df_reference,
        temp_df_production,
        lag_df_production,
    ) = group_features(static_data, production_data)

    static_temp_scores = {}
    normalized_static_temp_scores = {}
    static_lag_scores = {}
    normalized_static_lag_scores = {}

    temp_ref_groups = dict(tuple(temp_df_reference.groupby([SEASON_COL, HOUR_COL])))
    for bin_prod, group_prod in temp_df_production.groupby([SEASON_COL, HOUR_COL]):
        group_ref = temp_ref_groups.get(bin_prod)
        if group_ref is None or len(group_ref) < min_ref_samples_for_drift:
            group_ref = fallback_reference_absent(temp_df_reference, "temperature", bin_prod)
        if group_ref is None:
            continue

        static_temp_score = detect_drift_wasserstein(group_ref, group_prod)
        if static_temp_score is not None:
            static_temp_scores[bin_prod] = static_temp_score

            iqr_ref = group_ref
            if len(iqr_ref) < min_ref_samples_for_iqr:
                iqr_ref = fallback_reference_for_iqr_temperature(
                    temp_df_reference, "temperature", bin_prod
                )

            if iqr_ref is not None:
                iqr_value = (
                    iqr_ref[VALUE_COL].quantile(0.75)
                    - iqr_ref[VALUE_COL].quantile(0.25)
                )
                if iqr_value > 0:
                    normalized_static_temp_scores[bin_prod] = static_temp_score / iqr_value

    for lag_val in lag_df_reference[FEATURE_COL].dropna().unique():
        lag_ref, lag_prod = group_lag_features(lag_df_reference, lag_df_production, lag_val)
        lag_ref_groups = dict(tuple(lag_ref.groupby([DAYTYPE_COL, HOLIDAY_COL, HOUR_COL])))
        for bin_prod, group_prod in lag_prod.groupby([DAYTYPE_COL, HOLIDAY_COL, HOUR_COL]):
            group_ref = lag_ref_groups.get(bin_prod)
            if group_ref is None or len(group_ref) < min_ref_samples_for_drift:
                group_ref = fallback_reference_absent_lag(lag_ref, lag_val, bin_prod)
            if group_ref is None:
                continue

            static_lag_score = detect_drift_wasserstein(group_ref, group_prod)
            if static_lag_score is not None:
                lag_key = (lag_val, *bin_prod)
                static_lag_scores[lag_key] = static_lag_score

                iqr_ref = group_ref
                if len(iqr_ref) < min_ref_samples_for_iqr:
                    iqr_ref = fallback_reference_for_iqr_lag(lag_ref, lag_val, bin_prod)

                if iqr_ref is not None:
                    iqr_value = (
                        iqr_ref[VALUE_COL].quantile(0.75)
                        - iqr_ref[VALUE_COL].quantile(0.25)
                    )
                    if iqr_value > 0:
                        normalized_static_lag_scores[lag_key] = static_lag_score / iqr_value

    drift_scores["static"] = {
        "temperature": static_temp_scores,
        "temperature_normalized": normalized_static_temp_scores,
        "lag": static_lag_scores,
        "lag_normalized": normalized_static_lag_scores,
    }
    return drift_scores


def fallback_reference_absent(reference_data, feature_name, bin_prod):
    season_bin, hour_bin = bin_prod
    feature_ref = reference_data[reference_data[FEATURE_COL] == feature_name]

    candidates = [
        feature_ref[
            (feature_ref[SEASON_COL] == season_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[feature_ref[SEASON_COL] == season_bin],
        feature_ref[feature_ref[HOUR_COL] == hour_bin],
        feature_ref,
    ]

    for candidate in candidates:
        if (
            candidate is not None
            and not candidate.empty
            and len(candidate) >= min_ref_samples_for_drift
        ):
            return candidate

    return None


def fallback_reference_absent_lag(reference_data, lag_val, bin_prod):
    day_type_bin, is_holiday_bin, hour_bin = bin_prod
    feature_ref = reference_data[reference_data[FEATURE_COL] == lag_val]

    candidates = [
        feature_ref[
            (feature_ref[DAYTYPE_COL] == day_type_bin)
            & (feature_ref[HOLIDAY_COL] == is_holiday_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[
            (feature_ref[DAYTYPE_COL] == day_type_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[feature_ref[HOUR_COL] == hour_bin],
        feature_ref,
    ]

    for candidate in candidates:
        if (
            candidate is not None
            and not candidate.empty
            and len(candidate) >= min_ref_samples_for_drift
        ):
            return candidate

    return None


def recent_reference_drift(recent_data, production_data):
    required_columns = {
        FEATURE_COL,
        SEASON_COL,
        HOUR_COL,
        DAYTYPE_COL,
        HOLIDAY_COL,
        VALUE_COL,
    }
    missing_static = required_columns - set(recent_data.columns)
    missing_production = required_columns - set(production_data.columns)
    if missing_static:
        raise ValueError(f"Missing columns in static data: {missing_static}")
    if missing_production:
        raise ValueError(f"Missing columns in production data: {missing_production}")

    unknown = {
        feature
        for feature in production_data[FEATURE_COL].dropna().astype(str).unique()
        if feature.lower() not in set(features)
    }
    if unknown:
        raise ValueError(f"Unknown features in production data: {unknown}")


def fallback_reference_for_iqr_temperature(reference_data, feature_name, bin_prod):
    season_bin, hour_bin = bin_prod
    feature_ref = reference_data[reference_data[FEATURE_COL] == feature_name]

    candidates = [
        feature_ref[
            (feature_ref[SEASON_COL] == season_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[feature_ref[SEASON_COL] == season_bin],
        feature_ref[feature_ref[HOUR_COL] == hour_bin],
        feature_ref,
    ]

    for candidate in candidates:
        if candidate is None or candidate.empty or len(candidate) < min_ref_samples_for_iqr:
            continue

        iqr_value = candidate[VALUE_COL].quantile(0.75) - candidate[VALUE_COL].quantile(0.25)
        if iqr_value > 0:
            return candidate

    return None


def fallback_reference_for_iqr_lag(reference_data, lag_val, bin_prod):
    day_type_bin, is_holiday_bin, hour_bin = bin_prod
    feature_ref = reference_data[reference_data[FEATURE_COL] == lag_val]

    candidates = [
        feature_ref[
            (feature_ref[DAYTYPE_COL] == day_type_bin)
            & (feature_ref[HOLIDAY_COL] == is_holiday_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[
            (feature_ref[DAYTYPE_COL] == day_type_bin)
            & (feature_ref[HOUR_COL] == hour_bin)
        ],
        feature_ref[feature_ref[HOUR_COL] == hour_bin],
        feature_ref,
    ]

    for candidate in candidates:
        if candidate is None or candidate.empty or len(candidate) < min_ref_samples_for_iqr:
            continue

        iqr_value = candidate[VALUE_COL].quantile(0.75) - candidate[VALUE_COL].quantile(0.25)
        if iqr_value > 0:
            return candidate

    return None


def detect_drift_wasserstein(reference_data, production_data):
    if reference_data.empty or production_data.empty:
        return None

    reference_values = reference_data[VALUE_COL].dropna().values
    production_values = production_data[VALUE_COL].dropna().values
    if len(reference_values) == 0 or len(production_values) == 0:
        return None

    return wasserstein_distance(reference_values, production_values)


def check_consecutive_drifts(normalized_wasserstein_scores, threshold):
    drifted_columns = []
    for column, score in normalized_wasserstein_scores.items():
        if score > threshold:
            drifted_columns.append(column)

    if len(drifted_columns) >= 12:
        print("Data drift detected")
        for col in drifted_columns:
            print(f" - {col}")
    else:
        print("No significant consecutive data drifts detected.")


In [ ]:
if __name__ == "__main__":
    wasserstein_score_list = []
    static_reference_data = pd.read_csv(config["data_drift"]["reference_training_data"])
    recent_reference_data = pd.read_csv(config["data_drift"]["reference_recent_data"])
    production_data = pd.read_csv(config["data_drift"]["production_data"])
    static_drift = static_reference_drift(static_reference_data, production_data)
    recent_drift = recent_reference_drift(recent_reference_data, production_data)
    drift_scores_wasserstein_static = detect_drift_wasserstein(static_reference_data, production_data)
    drift_scores_wasserstein_recent = detect_drift_wasserstein(recent_reference_data, production_data)
    print("Data Drift Scores (Wasserstein Distance):")
    for column, score in drift_scores_wasserstein_static.items():
        print(f" - {column}: {score}")

    is_drift_static = check_consecutive_drifts(drift_scores_wasserstein_static, threshold = 0.9)
    is_drift_recent = check_consecutive_drifts(drift_scores_wasserstein_recent, threshold = 0.9)
    wasserstein_score_list.append(drift_scores_wasserstein_static)



In [40]:
df = pd.DataFrame({'FeatureType':["Temperature", "Temperature", "Temperature","Temperature","Temperature","Temperature", "Temperature", "Lag_2", "Lag_3","Temperature", "Lag_1", "Lag_2", "Lag_2", "Lag_3"],'Season_Bin': ["Winter", "Winter", "Summer", "Spring","Spring","Spring","Spring","NULL", "NULL", "Winter", "NULL", "NULL", "NULL", "NULL"], "Hour_Bin": ["Bin1", "Bin1", "Bin2", "Bin2", "Bin2","Bin2","Bin2", "Bin2","Bin2","Bin5","Bin2","Bin2","Bin1","Bin3"],
                   "Value": [8,9,9,18,13,19,12,189,261,280,188,560,512,404]})
df

,FeatureType,Season_Bin,Hour_Bin,Value
0,Temperature,Winter,Bin1,8
1,Temperature,Winter,Bin1,9
2,Temperature,Summer,Bin2,9
3,Temperature,Spring,Bin2,18
4,Temperature,Spring,Bin2,13
5,Temperature,Spring,Bin2,19
6,Temperature,Spring,Bin2,12
7,Lag_2,NULL,Bin2,189
8,Lag_3,NULL,Bin2,261
9,Temperature,Winter,Bin5,280


In [41]:
temp_df = df[df["FeatureType"] == "Temperature"]

In [48]:
for key,group in temp_df.groupby(["Season_Bin","Hour_Bin"]):
    print(f"Hour Bin: {key}")
    print(group)

Hour Bin: ('Spring', 'Bin2')
   FeatureType Season_Bin Hour_Bin  Value
3  Temperature     Spring     Bin2     18
4  Temperature     Spring     Bin2     13
5  Temperature     Spring     Bin2     19
6  Temperature     Spring     Bin2     12
Hour Bin: ('Summer', 'Bin2')
   FeatureType Season_Bin Hour_Bin  Value
2  Temperature     Summer     Bin2      9
Hour Bin: ('Winter', 'Bin1')
   FeatureType Season_Bin Hour_Bin  Value
0  Temperature     Winter     Bin1      8
1  Temperature     Winter     Bin1      9
Hour Bin: ('Winter', 'Bin5')
   FeatureType Season_Bin Hour_Bin  Value
9  Temperature     Winter     Bin5    280


In [44]:
lag_df = df[df["FeatureType"].str.contains("lag") | df["FeatureType"].str.contains("Lag")]

In [46]:
for lag_type in lag_df["FeatureType"].unique():
    print(f"Lag Type: {lag_type}")
    print(lag_df[lag_df["FeatureType"] == lag_type])

Lag Type: Lag_2
   FeatureType Season_Bin Hour_Bin  Value
7        Lag_2       NULL     Bin2    189
11       Lag_2       NULL     Bin2    560
12       Lag_2       NULL     Bin1    512
Lag Type: Lag_3
   FeatureType Season_Bin Hour_Bin  Value
8        Lag_3       NULL     Bin2    261
13       Lag_3       NULL     Bin3    404
Lag Type: Lag_1
   FeatureType Season_Bin Hour_Bin  Value
10       Lag_1       NULL     Bin2    188


In [22]:
df

,FeatureType,Season_Bin,Hour_Bin,Value
0,Temperature,Winter,Bin1,8
1,Temperature,Winter,Bin1,9


In [25]:
print(df[df["FeatureType"] == "Temperature"])

   FeatureType Season_Bin Hour_Bin  Value
0  Temperature     Winter     Bin1      8
1  Temperature     Winter     Bin1      9


In [27]:
print(df["FeatureType"] == "Temperature")

0    True
1    True
Name: FeatureType, dtype: bool


In [8]:
df_2 = pd.DataFrame({'FeatureType':["Temperature", "Temperature", "Temperature","Temperature","Temperature","Temperature", "Temperature", "Lag_2", "Lag_3","Temperature", "Lag_1", "Lag_2", "Lag_2", "Lag_3"],'Season_Bin': ["Winter", "Winter", "Summer", "Spring","Spring","Spring","Spring","NULL", "NULL", "Winter", "NULL", "NULL", "NULL", "NULL"], "Hour_Bin": ["Bin1", "Bin1", "Bin2", "Bin2", "Bin2","Bin2","Bin2", "Bin2","Bin2","Bin5","Bin2","Bin2","Bin1","Bin3"],
                   "Value": [8,9,9,18,13,19,12,189,261,280,188,560,512,404]})
df_2

,FeatureType,Season_Bin,Hour_Bin,Value
0,Temperature,Winter,Bin1,8
1,Temperature,Winter,Bin1,9
2,Temperature,Summer,Bin2,9
3,Temperature,Spring,Bin2,18
4,Temperature,Spring,Bin2,13
5,Temperature,Spring,Bin2,19
6,Temperature,Spring,Bin2,12
7,Lag_2,NULL,Bin2,189
8,Lag_3,NULL,Bin2,261
9,Temperature,Winter,Bin5,280


In [16]:
df_2_ref_groups = dict(tuple(df_2.groupby(["Season_Bin", "Hour_Bin"])))

In [17]:
print(df_2_ref_groups)

{('NULL', 'Bin1'):    FeatureType Season_Bin Hour_Bin  Value
12       Lag_2       NULL     Bin1    512, ('NULL', 'Bin2'):    FeatureType Season_Bin Hour_Bin  Value
7        Lag_2       NULL     Bin2    189
8        Lag_3       NULL     Bin2    261
10       Lag_1       NULL     Bin2    188
11       Lag_2       NULL     Bin2    560, ('NULL', 'Bin3'):    FeatureType Season_Bin Hour_Bin  Value
13       Lag_3       NULL     Bin3    404, ('Spring', 'Bin2'):    FeatureType Season_Bin Hour_Bin  Value
3  Temperature     Spring     Bin2     18
4  Temperature     Spring     Bin2     13
5  Temperature     Spring     Bin2     19
6  Temperature     Spring     Bin2     12, ('Summer', 'Bin2'):    FeatureType Season_Bin Hour_Bin  Value
2  Temperature     Summer     Bin2      9, ('Winter', 'Bin1'):    FeatureType Season_Bin Hour_Bin  Value
0  Temperature     Winter     Bin1      8
1  Temperature     Winter     Bin1      9, ('Winter', 'Bin5'):    FeatureType Season_Bin Hour_Bin  Value
9  Temperature     

In [18]:
print(df_2)

    FeatureType Season_Bin Hour_Bin  Value
0   Temperature     Winter     Bin1      8
1   Temperature     Winter     Bin1      9
2   Temperature     Summer     Bin2      9
3   Temperature     Spring     Bin2     18
4   Temperature     Spring     Bin2     13
5   Temperature     Spring     Bin2     19
6   Temperature     Spring     Bin2     12
7         Lag_2       NULL     Bin2    189
8         Lag_3       NULL     Bin2    261
9   Temperature     Winter     Bin5    280
10        Lag_1       NULL     Bin2    188
11        Lag_2       NULL     Bin2    560
12        Lag_2       NULL     Bin1    512
13        Lag_3       NULL     Bin3    404


In [19]:
df_2_ref_new = df_2_ref_groups.get(("Winter", "Bin1"))
print(df_2_ref_new)

   FeatureType Season_Bin Hour_Bin  Value
0  Temperature     Winter     Bin1      8
1  Temperature     Winter     Bin1      9
